# Загрузка данных и обучение модели

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score, cross_val_predict,
                                     GridSearchCV, train_test_split)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve, classification_report,
                             confusion_matrix, f1_score, average_precision_score, log_loss)
import optuna, warnings, plotly.graph_objects as go
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
df = pd.read_excel("../data/PG_GEO_Dataset.xlsx")
print(f"Размер датасета: {df.shape}")
print(f"Баланс классов:\n{df['cited_in_generative_answer'].value_counts()}")
print(f"Доля положительных: {df['cited_in_generative_answer'].mean():.2%}")

Размер датасета: (218, 62)
Баланс классов:
cited_in_generative_answer
1    110
0    108
Name: count, dtype: int64
Доля положительных: 50.46%


In [ ]:
TARGET = "cited_in_generative_answer"
META = ["query", "query_type", "product_category", "source_url", "domain", "source_type"]

feature_cols = [c for c in df.columns if c not in META + [TARGET]]
X = df[feature_cols].fillna(0).astype(float)
y = df[TARGET]

# Zero-variance
vt = VarianceThreshold(threshold=0.0)
X = pd.DataFrame(vt.fit_transform(X), columns=X.columns[vt.get_support()])
feature_cols = list(X.columns)

# Мультиколлинеарность
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.90)]
if to_drop:
    print(f"Убраны коррелированные: {to_drop}")
    X = X.drop(columns=to_drop)
    feature_cols = list(X.columns)

# Масштабирование + сплит
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Датасет: {len(df)} строк, {len(feature_cols)} фичей\n"
      f"Баланс: cited=1 → {y.sum()}, cited=0 → {(y==0).sum()}\n"
      f"Train: {len(y_train)} (pos={y_train.sum()})\n"
      f"Test:  {len(y_test)} (pos={y_test.sum()})")

Убраны коррелированные: ['has_numeric_data', 'has_recommendation', 'user_rating', 'has_sources_cited', 'is_serp_link', 'source_type_serp_internal']
Датасет: 218 строк, 49 фичей
Баланс: cited=1 → 110, cited=0 → 108
Train: 174 (pos=88)
Test:  44 (pos=22)


In [ ]:
from sklearn.feature_selection import RFECV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rfecv = RFECV(
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    step=1, cv=cv, scoring="roc_auc", min_features_to_select=5
)
rfecv.fit(X_train, y_train)

feature_cols = [f for f, s in zip(feature_cols, rfecv.support_) if s]
X_train = rfecv.transform(X_train)
X_test = rfecv.transform(X_test)

print(f"RFECV: оставлено {len(feature_cols)} фичей, AUC={rfecv.cv_results_['mean_test_score'].max():.4f}")

RFECV: оставлено 40 фичей, AUC=0.7503


# Baseline

In [ ]:
baseline = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)

auc = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="roc_auc")
f1 = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="f1")

print(f"BASELINE\n  ROC-AUC:  {auc.mean():.3f} ± {auc.std():.3f}\n"
      f"  F1:       {f1.mean():.3f} ± {f1.std():.3f}\n")

BASELINE
  ROC-AUC:  0.780 ± 0.089
  F1:       0.674 ± 0.097



#Optuna

In [ ]:
def objective(trial):
    penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])
    params = {
        "C": trial.suggest_float("C", 1e-4, 10.0, log=True),
        "penalty": penalty,
        "solver": "saga",
        "max_iter": 2000,
        "random_state": 42,
        "class_weight": trial.suggest_categorical("class_weight", ["balanced", "none"]),
    }
    if params["class_weight"] == "none":
        params["class_weight"] = None
    if penalty == "elasticnet":
        params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.0, 1.0)
    try:
        return cross_val_score(LogisticRegression(**params), X_train, y_train, cv=cv, scoring="roc_auc").mean()
    except:
        return 0.5

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"OPTUNA\n  Best AUC: {study.best_value:.4f}\n  Params:   {study.best_params}")

  0%|          | 0/100 [00:00<?, ?it/s]

OPTUNA
  Best AUC: 0.7844
  Params:   {'penalty': 'l2', 'C': 3.0170816450251206, 'class_weight': 'balanced'}


# Финальная модель

In [ ]:
# Лучшие параметры
bp = study.best_params.copy()
bp.update({"solver": "saga", "max_iter": 2000, "random_state": 42})
if bp.get("class_weight") == "none":
    bp["class_weight"] = None
if bp.get("penalty") != "elasticnet":
    bp.pop("l1_ratio", None)

print(f"Финальные параметры: {bp}")

model = LogisticRegression(**bp)

# CV на трейне
y_proba_cv = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
y_pred_cv = (y_proba_cv >= 0.5).astype(int)

print(f"\nTRAIN (CV):\n"
      f"  ROC-AUC:  {roc_auc_score(y_train, y_proba_cv):.4f}\n")
print(classification_report(y_train, y_pred_cv, target_names=["Не цитируется", "Цитируется"]))

# Тест
model.fit(X_train, y_train)
y_proba_test = model.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= 0.5).astype(int)

print(f"TEST:\n"
      f"  ROC-AUC:  {roc_auc_score(y_test, y_proba_test):.4f}\n")
print(classification_report(y_test, y_pred_test, target_names=["Не цитируется", "Цитируется"]))

# Коэффициенты
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coef": model.coef_[0],
    "OR": np.exp(model.coef_[0]),
}).assign(abs_coef=lambda d: d.coef.abs()).sort_values("abs_coef", ascending=False)

print("ТОП-15 фичей:")
for _, r in coef_df.head(15).iterrows():
    print(f"  {'+'if r.coef>0 else '-'} {r.feature:<40} coef={r.coef:+.3f}  OR={r.OR:.2f}")

Финальные параметры: {'penalty': 'l2', 'C': 3.0170816450251206, 'class_weight': 'balanced', 'solver': 'saga', 'max_iter': 2000, 'random_state': 42}

TRAIN (CV):
  ROC-AUC:  0.7772

               precision    recall  f1-score   support

Не цитируется       0.69      0.66      0.67        86
   Цитируется       0.68      0.70      0.69        88

     accuracy                           0.68       174
    macro avg       0.68      0.68      0.68       174
 weighted avg       0.68      0.68      0.68       174

TEST:
  ROC-AUC:  0.7417

               precision    recall  f1-score   support

Не цитируется       0.73      0.50      0.59        22
   Цитируется       0.62      0.82      0.71        22

     accuracy                           0.66        44
    macro avg       0.68      0.66      0.65        44
 weighted avg       0.68      0.66      0.65        44

ТОП-15 фичей:
  - covers_intent_fully                      coef=-1.448  OR=0.23
  - url_has_rating                           co

## Ансамбль моделей для сравнения

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier

ensemble = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(**bp)),
        ("gb", GradientBoostingClassifier(
            n_estimators=100, max_depth=3, subsample=0.8,
            learning_rate=0.05, random_state=42
        )),
    ],
    voting="soft"
)

# CV на трейне
auc_ens = cross_val_score(ensemble, X_train, y_train, cv=cv, scoring="roc_auc")
f1_ens = cross_val_score(ensemble, X_train, y_train, cv=cv, scoring="f1")
ll_ens = cross_val_score(ensemble, X_train, y_train, cv=cv, scoring="neg_log_loss")

print(f"ENSEMBLE (Train CV):\n"
      f"  ROC-AUC:  {auc_ens.mean():.3f} ± {auc_ens.std():.3f}\n"
      f"  F1:       {f1_ens.mean():.3f} ± {f1_ens.std():.3f}\n"
      f"  Log Loss: {-ll_ens.mean():.3f} ± {ll_ens.std():.3f}")

# Тест
ensemble.fit(X_train, y_train)
y_proba_ens = ensemble.predict_proba(X_test)[:, 1]
y_pred_ens = (y_proba_ens >= 0.5).astype(int)

print(f"\nENSEMBLE (Test):\n"
      f"  ROC-AUC:  {roc_auc_score(y_test, y_proba_ens):.4f}\n"
      f"  Log Loss: {log_loss(y_test, y_proba_ens):.4f}")
print(classification_report(y_test, y_pred_ens, target_names=["Не цитируется", "Цитируется"]))

ENSEMBLE (Train CV):
  ROC-AUC:  0.788 ± 0.094
  F1:       0.687 ± 0.102
  Log Loss: 0.564 ± 0.107

ENSEMBLE (Test):
  ROC-AUC:  0.7459
  Log Loss: 0.6051
               precision    recall  f1-score   support

Не цитируется       0.67      0.64      0.65        22
   Цитируется       0.65      0.68      0.67        22

     accuracy                           0.66        44
    macro avg       0.66      0.66      0.66        44
 weighted avg       0.66      0.66      0.66        44



In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(ensemble, X_test, y_test, n_repeats=30,
                              scoring="roc_auc", random_state=42)

ens_imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=False)

print("ENSEMBLE — ТОП-15 фичей:")
for _, r in ens_imp.head(15).iterrows():
    print(f"  {r.feature:<40} importance={r.importance:+.4f} ± {r['std']:.4f}")

ENSEMBLE — ТОП-15 фичей:
  url_has_rating                           importance=+0.0353 ± 0.0390
  qtype_category                           importance=+0.0275 ± 0.0255
  product_focus_ratio                      importance=+0.0254 ± 0.0274
  url_has_review                           importance=+0.0209 ± 0.0081
  cat_oral                                 importance=+0.0136 ± 0.0114
  covers_intent_fully                      importance=+0.0088 ± 0.0325
  content_freshness_2025                   importance=+0.0072 ± 0.0192
  has_faq                                  importance=+0.0068 ± 0.0295
  qtype_recommendation                     importance=+0.0065 ± 0.0115
  numeric_data_count                       importance=+0.0048 ± 0.0199
  has_pros_cons                            importance=+0.0042 ± 0.0115
  readability_score                        importance=+0.0037 ± 0.0311
  brand_mentioned                          importance=+0.0031 ± 0.0208
  source_type_expert_review                importanc

# Визуализация

In [ ]:
import os, kaleido
os.makedirs("plots", exist_ok=True)

# 6.1 Feature importance
top = coef_df.head(20).sort_values("coef")
fig = go.Figure(go.Bar(
    x=top.coef, y=top.feature, orientation="h",
    marker_color=["#2ecc71" if c > 0 else "#e74c3c" for c in top.coef],
    text=[f"OR={o:.2f}" for o in top.OR], textposition="inside",
    insidetextanchor="middle", textfont=dict(color="white", size=11),
))
fig.update_layout(
    title="Топ-20 коэффициентов",
    xaxis_title="Coefficient",
    height=650,
    margin=dict(l=280, r=30),
    xaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor="black"),
)
fig.write_image("plots/01_feature_importance.png", scale=2)
fig.show()

# 6.1b Ensemble feature importance
top_ens = ens_imp.head(20).sort_values("importance")
fig = go.Figure(go.Bar(
    x=top_ens.importance, y=top_ens.feature, orientation="h",
    error_x=dict(type="data", array=top_ens["std"]),
    marker_color="teal",
))
fig.update_layout(title="Ensemble: Топ-20 фичей (permutation importance)",
                  xaxis_title="ΔAUC при перемешивании", height=600, margin=dict(l=250))
fig.write_image("plots/01b_ensemble_importance.png", scale=2)
fig.show()

# 6.2 ROC — Train vs Test (LogReg + Ensemble)
fig = go.Figure()
for name, yt, yp in [("LR Train CV", y_train, y_proba_cv),
                      ("LR Test", y_test, y_proba_test),
                      ("Ensemble Test", y_test, y_proba_ens)]:
    fpr, tpr, _ = roc_curve(yt, yp)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{name} AUC={roc_auc_score(yt, yp):.3f}"))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash="dash", color="gray"), name="Random"))
fig.update_layout(title="ROC: LogReg vs Ensemble", xaxis_title="FPR", yaxis_title="TPR",
                  height=480, width=600)
fig.write_image("plots/02_roc_curve.png", scale=2)
fig.show()

# 6.3 Precision-Recall — Train vs Test
fig = go.Figure()
for name, yt, yp in [("LR Train CV", y_train, y_proba_cv),
                      ("LR Test", y_test, y_proba_test),
                      ("Ensemble Test", y_test, y_proba_ens)]:
    prec, rec, _ = precision_recall_curve(yt, yp)
    ap = average_precision_score(yt, yp)
    fig.add_trace(go.Scatter(x=rec, y=prec, name=f"{name} AP={ap:.3f}"))
fig.add_hline(y=y.mean(), line_dash="dash", line_color="gray", annotation_text="Baseline")
fig.update_layout(title="Precision-Recall", xaxis_title="Recall",
                  yaxis_title="Precision", height=480, width=600)
fig.write_image("plots/03_precision_recall.png", scale=2)
fig.show()

# 6.4 Confusion Matrix — LogReg Test
cm = confusion_matrix(y_test, y_pred_test)
fig = go.Figure(go.Heatmap(
    z=cm, x=["Pred 0", "Pred 1"], y=["True 0", "True 1"],
    text=cm, texttemplate="%{text}", colorscale="Blues", showscale=False,
))
fig.update_layout(title="Confusion Matrix — LogReg (Test)", height=400, width=400)
fig.write_image("plots/04_confusion_lr.png", scale=2)
fig.show()

# 6.5 Confusion Matrix — Ensemble Test
cm_ens = confusion_matrix(y_test, y_pred_ens)
fig = go.Figure(go.Heatmap(
    z=cm_ens, x=["Pred 0", "Pred 1"], y=["True 0", "True 1"],
    text=cm_ens, texttemplate="%{text}", colorscale="Greens", showscale=False,
))
fig.update_layout(title="Confusion Matrix — Ensemble (Test)", height=400, width=400)
fig.write_image("plots/05_confusion_ensemble.png", scale=2)
fig.show()

# 6.6 Threshold sweep (Test) — LogReg
thresholds = np.arange(0.1, 0.91, 0.01)
f1s = [f1_score(y_test, (y_proba_test >= t).astype(int), zero_division=0) for t in thresholds]
best_t = thresholds[np.argmax(f1s)]

fig = go.Figure(go.Scatter(x=thresholds, y=f1s, line=dict(color="crimson")))
fig.add_vline(x=best_t, line_dash="dash", annotation_text=f"Best={best_t:.2f}")
fig.update_layout(title="F1 vs Threshold (Test)", xaxis_title="Threshold",
                  yaxis_title="F1", height=400)
fig.write_image("plots/06_threshold.png", scale=2)
fig.show()

# 6.7 AUC по категориям и типам запросов
train_idx = y_train.index
for i, (col, label) in enumerate([("product_category", "Категория"), ("query_type", "Тип запроса")]):
    names, aucs = [], []
    for val in df[col].unique():
        mask = df.loc[train_idx, col] == val
        if mask.sum() < 20: continue
        s = cross_val_score(LogisticRegression(**bp), X_train[mask], y_train[mask],
                            cv=StratifiedKFold(3, shuffle=True, random_state=42), scoring="roc_auc")
        names.append(val); aucs.append(s.mean())
    fig = go.Figure(go.Bar(x=names, y=aucs, marker_color="steelblue"))
    fig.add_hline(y=0.5, line_dash="dash", line_color="red")
    fig.update_layout(title=f"AUC по: {label}", yaxis_range=[0, 1.05], height=400)
    fig.write_image(f"plots/07_auc_{col}.png", scale=2)
    fig.show()

# 6.8 Optuna history
vals = [t.value for t in study.trials]
fig = go.Figure()
fig.add_trace(go.Scatter(y=vals, mode="markers", name="Trial", marker=dict(size=4, opacity=0.5)))
fig.add_trace(go.Scatter(y=np.maximum.accumulate(vals), name="Best", line=dict(color="crimson")))
fig.update_layout(title="Optuna: история", xaxis_title="Trial", yaxis_title="AUC", height=400)
fig.write_image("plots/08_optuna_history.png", scale=2)
fig.show()

print(f"Все графики сохранены в папку plots/")

Все графики сохранены в папку plots/
